In [ ]:
import os
import re
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

RUN_EVAL = True
FORCE_REEVAL = False
PARALLEL_WORKERS = 4

PROJECT_ROOT = os.path.abspath("..")
RECON_ROOT = os.path.join(PROJECT_ROOT, "outputs/reconstruction")
GT_ROOT = os.environ.get("DATASET_ROOT", "path/to/your/image_pairs")
EVAL_BIN = os.environ.get(
    "MULTIVIEW_EVAL_BIN",
    os.path.abspath("path/to/your/multiview_evaluator"),
)
EVAL_FILE = os.path.abspath("evaluation_results.txt")

TOLS = "0.01,0.02,0.05,0.08,0.1,0.14,0.2,0.3,0.4,0.5,0.7,1.0"

SCENES = [
    (name, name)
    for name in sorted(os.listdir(RECON_ROOT))
    if os.path.isdir(os.path.join(RECON_ROOT, name))
]

MODELS = [
    "zhou_raw_points.ply",
    "zhou_curve_points.ply",
    "dlt_curve_points.ply",
    "ours_curve_points.ply",
]
MODEL_LABELS = {
    "zhou_raw_points.ply": "Zhou et al.",
    "zhou_curve_points.ply": "Zhou et al. + NURBS",
    "dlt_curve_points.ply": "CSD (ours) + DLT + NURBS",
    "ours_curve_points.ply": "Proposed",
}
MODEL_KEYS = {
    "zhou_raw_points.ply": "zhou_raw",
    "zhou_curve_points.ply": "zhou_curve",
    "dlt_curve_points.ply": "dlt_curve",
    "ours_curve_points.ply": "ours_curve",
}

METRIC_RE = re.compile(r"^(Tolerances|Completenesses|Accuracies|F1-scores):")

def extract_metric_lines(stdout: str):
    lines = [ln.strip() for ln in stdout.splitlines() if METRIC_RE.match(ln.strip())]
    if len(lines) != 4:
        raise RuntimeError(
            f"Expected four metric lines; got {len(lines)}\n--- tail ---\n{stdout[-800:]}"
        )
    return lines

def run_single_eval(recon_path, gt_path):
    if not os.path.isfile(EVAL_BIN):
        raise FileNotFoundError(
            f"Evaluation executable not found: {EVAL_BIN}\nSet MULTIVIEW_EVAL_BIN"
        )
    proc = subprocess.run(
        [
            EVAL_BIN,
            "--reconstruction_ply_path", recon_path,
            "--ground_truth_mlp_path", gt_path,
            "--tolerances", TOLS,
        ],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        err = proc.stderr.strip() or proc.stdout.strip()
        raise RuntimeError(f"exit {proc.returncode}: {err[-500:]}")
    return extract_metric_lines(proc.stdout)

def eval_one_job(job_idx, scene, recon_dir, name):
    """Return the job index, status, message and metrics."""
    recon_path = os.path.join(RECON_ROOT, recon_dir, name)
    gt_path = os.path.join(GT_ROOT, recon_dir, "scan_clean/scan_alignment.mlp")

    if not os.path.isfile(recon_path):
        return job_idx, "skip", f"Missing reconstruction: {recon_path}", None
    if not os.path.isfile(gt_path):
        return job_idx, "skip", f"Missing ground truth: {gt_path}", None

    try:
        lines = run_single_eval(recon_path, gt_path)
        return job_idx, "ok", None, (scene, name, lines)
    except Exception as exc:
        return job_idx, "fail", f"{scene}/{name}: {exc}", None

def run_all_evaluations():
    jobs = [
        (scene, recon_dir, name)
        for scene, recon_dir in SCENES
        for name in MODELS
    ]
    n_jobs = len(jobs)
    results = [None] * n_jobs
    skipped = []

    workers = max(1, int(PARALLEL_WORKERS))
    mode = f"{workers} workers" if workers > 1 else "serial"
    print(f"Evaluating {n_jobs} jobs ({mode})...")

    if workers == 1:
        for job_idx, (scene, recon_dir, name) in enumerate(
            tqdm(jobs, desc="Point cloud evaluation", unit="job")
        ):
            _, status, msg, payload = eval_one_job(job_idx, scene, recon_dir, name)
            if status == "ok":
                results[job_idx] = payload
            else:
                skipped.append(f"{'Skipped' if status == 'skip' else 'Failed'} ({msg})")
    else:
        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = [
                executor.submit(eval_one_job, job_idx, scene, recon_dir, name)
                for job_idx, (scene, recon_dir, name) in enumerate(jobs)
            ]
            for fut in tqdm(as_completed(futures), total=n_jobs, desc="Point cloud evaluation", unit="job"):
                job_idx, status, msg, payload = fut.result()
                if status == "ok":
                    results[job_idx] = payload
                else:
                    skipped.append(f"{'Skipped' if status == 'skip' else 'Failed'} ({msg})")

    with open(EVAL_FILE, "w") as out:
        out.write("===== Evaluation Results =====\n")
        for job_idx, (scene, recon_dir, name) in enumerate(jobs):
            payload = results[job_idx]
            if payload is None:
                continue
            out_scene, out_name, lines = payload
            out.write(f"[{out_scene}/{out_name}]\n")
            for line in lines:
                out.write(line + "\n")
            out.write("\n")

    ok_count = sum(r is not None for r in results)
    print(f"\nEvaluation complete: {ok_count}/{n_jobs} succeeded; saved to {EVAL_FILE}")
    if skipped:
        print(f"Skipped/Failed {len(skipped)} items:")
        for msg in skipped:
            print(f"   {msg}")

if RUN_EVAL:
    if os.path.exists(EVAL_FILE) and not FORCE_REEVAL:
        print(f"Results exist: {EVAL_FILE}; set FORCE_REEVAL=True to rerun")
    else:
        run_all_evaluations()
else:
    print(f"RUN_EVAL=False; evaluation skipped")

if not os.path.exists(EVAL_FILE):
    raise FileNotFoundError(f"Missing {EVAL_FILE}; set RUN_EVAL=True or generate it first")


In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

if "EVAL_FILE" not in globals():
    raise RuntimeError("Run the configuration and evaluation cell first")

data = defaultdict(lambda: defaultdict(lambda: {"Completenesses": [], "Accuracies": [], "F1-scores": []}))

current_model = None
tolerances = []

with open(EVAL_FILE, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("[") and line.endswith("]"):
            current_model = line[1:-1].split("/")[-1]
        elif line.startswith("Tolerances:"):
            tolerances = [float(x) for x in line.split()[1:]]
        elif line.startswith("Completenesses:"):
            values = [float(x) for x in line.split()[1:]]
            for t, v in zip(tolerances, values):
                data[current_model][t]["Completenesses"].append(v)
        elif line.startswith("Accuracies:"):
            values = [float(x) for x in line.split()[1:]]
            for t, v in zip(tolerances, values):
                data[current_model][t]["Accuracies"].append(v)
        elif line.startswith("F1-scores:"):
            values = [float(x) for x in line.split()[1:]]
            for t, v in zip(tolerances, values):
                data[current_model][t]["F1-scores"].append(v)

rows = []
for model, tol_dict in data.items():
    if model not in MODELS:
        continue
    for tol, metrics in tol_dict.items():
        rows.append({
            "Model": model,
            "Label": MODEL_LABELS[model],
            "Tolerance": tol,
            "Completeness_avg": np.mean(metrics["Completenesses"]),
            "Accuracy_avg": np.mean(metrics["Accuracies"]),
            "F1-score_avg": np.mean(metrics["F1-scores"]),
        })

df_avg = pd.DataFrame(rows)
df_avg = df_avg.sort_values(["Model", "Tolerance"]).reset_index(drop=True)

missing = set(MODELS) - set(df_avg["Model"].unique())
if missing:
    raise ValueError(f"Missing models in evaluation results: {sorted(missing)}")

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(df_avg)

out_csv = os.path.abspath("metrics_per_model.csv")
df_avg.to_csv(out_csv, index=False)
print(f"Saved {out_csv}")


In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")

metrics = ["Completeness_avg", "Accuracy_avg", "F1-score_avg"]
y_labels = ["Recall (Completeness)", "Precision (Accuracy)", "F1-score"]

for metric, ylabel in zip(metrics, y_labels):
    plt.figure(figsize=(8, 5))
    for model in MODELS:
        df_model = df_avg[df_avg["Model"] == model]
        plt.plot(
            df_model["Tolerance"],
            df_model[metric],
            marker="o",
            label=MODEL_LABELS[model],
        )
    plt.xlabel("Tolerance (m)")
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} vs Tolerance")
    plt.xticks(df_model["Tolerance"], df_model["Tolerance"])
    plt.legend()
    plt.grid(True, which="both", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()


In [ ]:
RUNTIME_PLACEHOLDER = -1.0
RUNTIME_TXT_PATH = os.path.join(PROJECT_ROOT, "outputs/reconstruction/runtimes.txt")

PAPER_RUNTIME_COLUMNS = {
    "zhou_raw": "zhou_raw",
    "zhou_curve": "zhou_total",
    "dlt_curve": "dlt_total",
    "ours_curve": "ours_all_total",
}

def mean_runtime_ms(path, column):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Run reconstruction to generate {path}")
    values = []
    with open(path, "r") as f:
        header = f.readline().split()
        if column not in header:
            raise ValueError(f"{path} Missing column {column}; header: {header}")
        col_idx = header.index(column)
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) <= col_idx:
                continue
            val = float(parts[col_idx])
            if val != RUNTIME_PLACEHOLDER:
                values.append(val)
    if not values:
        return float("nan")
    return float(np.mean(values))

PAPER_RUNTIME_MS = {
    key: mean_runtime_ms(RUNTIME_TXT_PATH, col)
    for key, col in PAPER_RUNTIME_COLUMNS.items()
}

print(f"Mean runtime ({RUNTIME_TXT_PATH}):")
for key, col in PAPER_RUNTIME_COLUMNS.items():
    val = PAPER_RUNTIME_MS[key]
    if np.isfinite(val):
        print(f"  {key} ({col}): {val:.1f} ms")
    else:
        print(f"  {key} ({col}): N/A")


In [ ]:
PAPER_THRESHOLDS = (0.10, 0.50, 1.00)
PAPER_TABLE_METHODS = [
    ("zhou_raw_points.ply", "zhou_raw", "Zhou et al."),
    ("zhou_curve_points.ply", "zhou_curve", "Zhou et al. + NURBS"),
    ("dlt_curve_points.ply", "dlt_curve", "CSD (ours) + DLT + NURBS"),
    ("ours_curve_points.ply", "ours_curve", "Proposed"),
]

if "PAPER_RUNTIME_MS" not in globals():
    raise RuntimeError("Run the runtime cell first")

output_path = os.path.abspath("metrics_table.csv")

def fmt_pct(val):
    return f"{val * 100:.1f}"

def fmt_time_ms(val):
    return "N/A" if not np.isfinite(val) else f"{val:.1f}"

def lookup(df, model, tol):
    row = df[(df["Model"] == model) & (np.isclose(df["Tolerance"], tol))]
    if row.empty:
        raise ValueError(f"Missing {model} @ tolerance={tol}")
    return row.iloc[0]

header = [
    "method", "label",
    "P0.1", "R0.1", "F0.1",
    "P0.5", "R0.5", "F0.5",
    "P1.0", "R1.0", "F1.0",
    "time_ms",
]

table_rows = []
with open(output_path, "w", newline="") as f:
    import csv
    writer = csv.writer(f)
    writer.writerow(header)
    for model, key, label in PAPER_TABLE_METHODS:
        rows_by_tol = {t: lookup(df_avg, model, t) for t in PAPER_THRESHOLDS}
        out_row = [
            key, label,
            fmt_pct(rows_by_tol[0.10]["Accuracy_avg"]),
            fmt_pct(rows_by_tol[0.10]["Completeness_avg"]),
            fmt_pct(rows_by_tol[0.10]["F1-score_avg"]),
            fmt_pct(rows_by_tol[0.50]["Accuracy_avg"]),
            fmt_pct(rows_by_tol[0.50]["Completeness_avg"]),
            fmt_pct(rows_by_tol[0.50]["F1-score_avg"]),
            fmt_pct(rows_by_tol[1.00]["Accuracy_avg"]),
            fmt_pct(rows_by_tol[1.00]["Completeness_avg"]),
            fmt_pct(rows_by_tol[1.00]["F1-score_avg"]),
            fmt_time_ms(PAPER_RUNTIME_MS[key]),
        ]
        writer.writerow(out_row)
        table_rows.append(out_row)

print(f"Exported {output_path}\n")
print(f"{'Method':<35} {'P0.1':>6} {'R0.1':>6} {'F0.1':>6} {'P0.5':>6} {'R0.5':>6} {'F0.5':>6} {'P1.0':>6} {'R1.0':>6} {'F1.0':>6} {'Time↓':>8}")
print("-" * 98)
for row in table_rows:
    print(f"{row[1]:<35} {row[2]:>6} {row[3]:>6} {row[4]:>6} {row[5]:>6} {row[6]:>6} {row[7]:>6} {row[8]:>6} {row[9]:>6} {row[10]:>6} {row[11]:>8}")
